In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 67
pd.set_option("display.max_columns", None)

df = pd.read_csv("../../H2_cleaned.csv", dtype={"Agent": str})
print(df.shape)

(48991, 31)


Column selection for ADR

Prediction point: the price is predicted at the moment the booking is made. A column is only allowed as a predictor if its value is known at that moment and it is related to price.

Leakage (only known after the booking is made) Drop
- AssignedRoomType: room actually given at check-in, upgrades are decided after booking
- ReservationStatus: final outcome (Check-Out / Canceled / No-Show)
- ReservationStatusDate: date of that final outcome, always after the booking
- IsCanceled: cancellation happens later in the booking lifecycle
- BookingChanges: amendments after booking, a change of dates or room can change the price
- RequiredCarParkingSpaces: recorded when the guest arrives, not at booking time
- DaysInWaitingList: keeps counting after the booking enters the PMS so not fully known at booking
- TotalOfSpecialRequests: guests can add requests after booking

Redundant Drop
- ArrivalDateWeekNumber: same seasonal info as ArrivalDateMonth
- ArrivalDateDayOfMonth: day number alone does not explain price

Too many IDs Drop
- Agent: 219 agent IDs, most with very few bookings (same decision as the classifier)

In [2]:
drop_cols = [
    "AssignedRoomType", "ReservationStatus", "ReservationStatusDate", "IsCanceled", "BookingChanges", "RequiredCarParkingSpaces", "DaysInWaitingList", "TotalOfSpecialRequests", "ArrivalDateWeekNumber", "ArrivalDateDayOfMonth", "Agent",
]

df = df.drop(columns=drop_cols)
print("dropped:", len(drop_cols), " remaining:", df.shape[1])
print("predictors:", [c for c in df.columns if c != "ADR"])

dropped: 11  remaining: 20
predictors: ['LeadTime', 'ArrivalDateYear', 'ArrivalDateMonth', 'StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children', 'Babies', 'Meal', 'Country', 'MarketSegment', 'DistributionChannel', 'IsRepeatedGuest', 'PreviousCancellations', 'PreviousBookingsNotCanceled', 'ReservedRoomType', 'DepositType', 'CustomerType', 'HasCompany']


Invalid rows

- ADR = 0: free stays are not a price (see eda.ipynb section 1) **Drop the rows**
- Undefined MarketSegment / DistributionChannel: missing info (same as the classifier) **Drop the rows**

In [3]:
before = len(df)
df = df[df["ADR"] > 0]
print("ADR = 0 removed:", before - len(df))

before = len(df)
df = df[(df["MarketSegment"] != "Undefined") & (df["DistributionChannel"] != "Undefined")]
print("Undefined removed:", before - len(df))
print("rows left:", len(df))

ADR = 0 removed: 49
Undefined removed: 2
rows left: 48940


Preprocessing pipeline

| Columns | Step |
|---|---|
| Categorical | One-hot, rare categories (< 0.5% of rows) grouped into one "infrequent" column |
| LeadTime, stay durations | log(1 + x) + scaling |
| other numerics | Scaling |
| IsRepeatedGuest, HasCompany | already 0/1, kept as they are |

The pipeline is only **defined** here. It gets fitted on the training set after the split so no test-set information leaks into the scaling or the encoding.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

TARGET = "ADR"
X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = ["ReservedRoomType", "Meal", "Country", "MarketSegment", "DistributionChannel",
            "CustomerType", "DepositType", "ArrivalDateMonth"]
skewed_cols = ["LeadTime", "StaysInWeekendNights", "StaysInWeekNights"]
num_cols = ["Adults", "Children", "Babies", "ArrivalDateYear",
            "PreviousCancellations", "PreviousBookingsNotCanceled"]
binary_cols = ["IsRepeatedGuest", "HasCompany"]

groups = cat_cols + skewed_cols + num_cols + binary_cols
assert sorted(groups) == sorted(X.columns), set(X.columns) ^ set(groups)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.005), cat_cols),
    ("skewed", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                         ("sc", StandardScaler())]), skewed_cols),
    ("num", StandardScaler(), num_cols),
    ("bin", "passthrough", binary_cols),
])

print("categorical:", cat_cols)
print("skewed     :", skewed_cols)
print("numerical  :", num_cols)
print("binary     :", binary_cols)


categorical: ['ReservedRoomType', 'Meal', 'Country', 'MarketSegment', 'DistributionChannel', 'CustomerType', 'DepositType', 'ArrivalDateMonth']
skewed     : ['LeadTime', 'StaysInWeekendNights', 'StaysInWeekNights']
numerical  : ['Adults', 'Children', 'Babies', 'ArrivalDateYear', 'PreviousCancellations', 'PreviousBookingsNotCanceled']
binary     : ['IsRepeatedGuest', 'HasCompany']


Save the cleaned regression data (columns dropped, invalid rows removed). Encoding and scaling are not applied here, they are fitted on the training set after the split.

In [5]:
df.to_csv("../Data/H2_regression.csv", index=False)
print("saved H2_regression.csv:", df.shape)

saved H2_regression.csv: (48940, 20)
